# PROGRAMA DE INGENIERÍA DE SISTEMAS Y COMPUTACIÓN
## TEORÍA DE LENGUAJES FORMALES — PARCIAL 1
### Simulación Computacional: Reconocedor de Placas Vehiculares (Colombia)

**Integrantes:**
* Nikoll Morales Londoño
* Felipe Valencia Londoño

**Temática 23:** Validador y reconocedor de placas vehiculares automotrices según normativa nacional colombiana (3 letras mayúsculas seguidas de 3 dígitos numéricos: `LLLDDD`).

> **Restricción Formal:** Implementación en Python puro sin dependencias externas ni librerías de expresiones regulares (`import re` está estrictamente prohibido).

## 1. Definición Formal de la Tupla del AFD

El Autómata Finito Determinista se define como la 5-tupla:
$$M = (Q, \Sigma, \delta, q_0, F)$$

- **$Q = \{q_0, q_1, q_2, q_3, q_4, q_5, q_6, q_E\}$:** Conjunto de 8 estados (incluyendo estado trampa $q_E$).
- **$\Sigma = \Lambda \cup \Delta$:** Alfabeto de 36 símbolos (26 letras mayúsculas $\Lambda$ y 10 dígitos $\Delta$).
- **$q_0$:** Estado inicial.
- **$F = \{q_6\}$:** Conjunto unitario de aceptación.
- **$\delta: Q \times \Sigma \to Q$:** Función de transición total.

In [ ]:
# Alfabeto formal base Sigma = Lambda U Delta
LETRAS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ")  # Lambda (|Lambda| = 26)
DIGITOS = set("0123456789")                  # Delta (|Delta| = 10)
ALFABETO = LETRAS.union(DIGITOS)             # Sigma (|Sigma| = 36)

# Estados formales
ESTADO_INICIAL = "q0"
ESTADOS_ACEPTACION = {"q6"}
ESTADO_TRAMPA = "qE"

DESCRIPCION_ESTADOS = {
    "q0": "Inicio: Esperando primera letra (L1)",
    "q1": "Leyó 1 letra válida: Esperando segunda letra (L2)",
    "q2": "Leyó 2 letras válidas: Esperando tercera letra (L3)",
    "q3": "Leyó 3 letras válidas: Esperando primer dígito (D1)",
    "q4": "Leyó 3 letras y 1 dígito: Esperando segundo dígito (D2)",
    "q5": "Leyó 3 letras y 2 dígitos: Esperando tercer dígito (D3)",
    "q6": "Estado de Aceptación: Placa completa (3 letras + 3 dígitos)",
    "qE": "Estado Trampa / Error: Desvío sintáctico irreversible"
}

## 2. Implementación de la Clase `AFDPlacas` y Función de Transición $\delta(q, a)$

In [ ]:
class AFDPlacas:
    """
    Simulador formal de AFD para placas de automotores en Colombia.
    """
    def __init__(self):
        self.q0 = ESTADO_INICIAL
        self.F = ESTADOS_ACEPTACION
        self.qE = ESTADO_TRAMPA
        self.sigma = ALFABETO

    def clasificar_simbolo(self, simbolo: str) -> str:
        if simbolo in LETRAS:
            return "L"
        elif simbolo in DIGITOS:
            return "D"
        else:
            return "OTHER"

    def delta(self, estado_actual: str, simbolo: str) -> str:
        tipo = self.clasificar_simbolo(simbolo)
        
        if tipo == "OTHER":
            return self.qE

        if estado_actual == "q0":
            return "q1" if tipo == "L" else self.qE
        elif estado_actual == "q1":
            return "q2" if tipo == "L" else self.qE
        elif estado_actual == "q2":
            return "q3" if tipo == "L" else self.qE
        elif estado_actual == "q3":
            return "q4" if tipo == "D" else self.qE
        elif estado_actual == "q4":
            return "q5" if tipo == "D" else self.qE
        elif estado_actual == "q5":
            return "q6" if tipo == "D" else self.qE
        elif estado_actual == "q6":
            return self.qE
        elif estado_actual == self.qE:
            return self.qE
        return self.qE

    def procesar_cadena(self, cadena: str) -> dict:
        estado_actual = self.q0
        traza = []
        estados_recorridos = [estado_actual]

        for i, simbolo in enumerate(cadena, start=1):
            clase = self.clasificar_simbolo(simbolo)
            estado_siguiente = self.delta(estado_actual, simbolo)
            
            if clase == "OTHER":
                motivo = f"Símbolo '{simbolo}' NO pertenece al alfabeto Sigma"
            elif estado_siguiente == self.qE and estado_actual != self.qE:
                if estado_actual in {"q0", "q1", "q2"}:
                    motivo = f"Esperaba Letra (L), pero llegó Dígito '{simbolo}'"
                elif estado_actual in {"q3", "q4", "q5"}:
                    motivo = f"Esperaba Dígito (D), pero llegó Letra '{simbolo}'"
                elif estado_actual == "q6":
                    motivo = f"Exceso de longitud (|w| > 6). Símbolo extra '{simbolo}'"
                else:
                    motivo = "Transición inválida a trampa"
            elif estado_actual == self.qE:
                motivo = f"Permanece en trampa qE con '{simbolo}'"
            else:
                motivo = f"Transición válida: {DESCRIPCION_ESTADOS.get(estado_siguiente, '')}"

            traza.append({
                "paso": i,
                "simbolo": simbolo,
                "clase": clase,
                "origen": estado_actual,
                "destino": estado_siguiente,
                "motivo": motivo
            })
            estado_actual = estado_siguiente
            estados_recorridos.append(estado_actual)

        es_aceptada = (estado_actual in self.F)
        diagnostico = "Placa ACEPTADA formalmente" if es_aceptada else f"Cadena RECHAZADA (finalizó en {estado_actual})"

        return {
            "cadena": cadena,
            "estados_recorridos": estados_recorridos,
            "traza": traza,
            "estado_final": estado_actual,
            "es_aceptada": es_aceptada,
            "diagnostico": diagnostico
        }

## 3. Función Auxiliar de Formateo y Visualización de la Traza

In [ ]:
def visualizar_traza(resultado: dict):
    cadena = resultado["cadena"]
    print(f"\n{'='*75}")
    print(f"ANÁLISIS DE LA CADENA: \"{cadena}\" (Longitud: {len(cadena)})")
    print(f"{'='*75}")
    
    if len(cadena) == 0:
        print(" [Cadena vacía lambda]: Se mantiene en q0")
    else:
        print(f"{'Paso':<6}{'Símb.':<8}{'Clase':<8}{'Origen':<10}{'Destino':<10}{'Detalle':<35}")
        print("-"*75)
        for p in resultado["traza"]:
            print(f"{p['paso']:<6}{p['simbolo']:<8}{p['clase']:<8}{p['origen']:<10}{p['destino']:<10}{p['motivo'][:34]:<35}")
            
    print("-"*75)
    print(f"Ruta de estados: {' -> '.join(resultado['estados_recorridos'])}")
    print(f"Estado final: {resultado['estado_final']}")
    veredicto = "ACEPTADA" if resultado["es_aceptada"] else "RECHAZADA"
    print(f"RESULTADO FINAL: [ {veredicto} ]")
    print(f"Diagnóstico: {resultado['diagnostico']}\n")

## 4. Batería de Pruebas Automatizada (15 Casos)

In [ ]:
casos_prueba = [
    ("ABC123", True, "Estructura estándar correcta (3 letras y 3 dígitos)"),
    ("KLR890", True, "Combinación intermedia válida"),
    ("ZZZ000", True, "Límite superior alfabético y dígitos en cero"),
    ("XYZ999", True, "Límite superior de dígitos (999)"),
    ("MNO541", True, "Transición adecuada letra-dígito"),
    ("AAA111", True, "Secuencia repetitiva homogénea"),
    ("abc123", False, "Minúsculas (fuera de Sigma)"),
    ("AB1234", False, "Falta una letra y sobra un dígito (2 letras + 4 dígitos)"),
    ("ABC1234", False, "Longitud 7 (excede 6 caracteres)"),
    ("ABC12", False, "Longitud 5 (incompleta)"),
    ("123ABC", False, "Orden invertido (dígitos antes de letras)"),
    ("AB-123", False, "Carácter separador '-' ajeno a Sigma"),
    ("ABC12D", False, "Posición 6 es letra en lugar de dígito"),
    ("", False, "Cadena vacía lambda"),
    ("AÑC123", False, "Contiene letra 'Ñ' ajena a Sigma")
]

afd = AFDPlacas()
exitos = 0

print(f"{'#':<4}{'Cadena':<10}{'Esperado':<12}{'Obtenido':<12}{'Estado Final':<15}{'Veredicto':<10}")
print("="*65)

for i, (cadena, esperado, desc) in enumerate(casos_prueba, start=1):
    res = afd.procesar_cadena(cadena)
    obtenido = res["es_aceptada"]
    ok = (obtenido == esperado)
    if ok:
        exitos += 1
    
    esp_str = "ACEPTADA" if esperado else "RECHAZADA"
    obt_str = "ACEPTADA" if obtenido else "RECHAZADA"
    ver_str = "OK" if ok else "FALLA"
    print(f"{i:<4}{cadena:<10}{esp_str:<12}{obt_str:<12}{res['estado_final']:<15}{ver_str:<10}")

print("="*65)
print(f"Resumen: {exitos}/{len(casos_prueba)} pruebas superadas con éxito ({(exitos/len(casos_prueba))*100:.1f}%).")

## 5. Prueba Interactiva de Ejemplo con Traza Detallada

In [ ]:
# Prueba de caso válido
visualizar_traza(afd.procesar_cadena("ABC123"))

# Prueba de caso inválido por exceso de longitud
visualizar_traza(afd.procesar_cadena("ABC1234"))

# Prueba de caso inválido por minúsculas
visualizar_traza(afd.procesar_cadena("abc123"))